# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


## Imports

In [ ]:
import math
import tempfile
import time
from pathlib import Path

import torch
from torch import Tensor, nn, tensor
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

import mlflow
from common.git import get_branch, get_sha
from common.model_registry import TRACKING_URI
from time_series.store_sales import MSLELoss, StoreData

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

## mlflow Setup

In [ ]:
mlflow.pytorch.autolog()  # pyright: ignore[reportPrivateImportUsage]
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("StoreSales_TransformerBasic")
mlflow.set_experiment_tags(
    {
        "dataset": "store-sales-kaggle",
        "task": "time-series-forecasting",
        "loss": "RMSLE",
        "prediction_horizon_days": "16",
    }
)



# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData(dtype=torch.float32)
store_data_loader = DataLoader(store_data, batch_size=10)
store_data.train

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

In [ ]:
class StoreSalesTransformer(nn.Module):
    def __init__(
        self,
        n_output_steps: int = store_data.output_lags,
        embedding_size: int = store_data.families.size * store_data.stores.shape[0],
        d_model: int = 128,
        nhead: int = 4,
    ) -> None:
        super().__init__()
        self.embedding_size = embedding_size
        self.n_output_steps = n_output_steps
        self.input_transform = nn.Linear(self.embedding_size, d_model)
        self.output_transform = nn.Linear(d_model, self.embedding_size)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True,
        )
        self.output_relu = nn.ReLU()

    def forward(self, input: Tensor) -> Tensor:
        input_internal = self.input_transform(input.flatten(-2))
        tgt_internal = torch.zeros(
            input_internal.shape[0],
            self.n_output_steps,
            self.transformer.d_model,
            dtype=input_internal.dtype,
            device=input_internal.device,
        )
        output_internal = self.transformer(input_internal, tgt_internal)
        out_shape = (-1, self.n_output_steps, *input.shape[-2:])
        return self.output_relu(
            self.output_transform(output_internal).reshape(out_shape)
        )


model = StoreSalesTransformer()
model

In [ ]:
batch_X, batch_y = next(iter(store_data_loader))
model(batch_X).shape

# Training

In [ ]:
epochs = 500
save_every_n_epochs = 50
learning_rate = 1e-3
split_fraction = 0.9
batch_size = 128

split_loc = int(len(store_data) * split_fraction)

train_data = Subset(store_data, range(0, split_loc))
val_data = Subset(store_data, range(split_loc, len(store_data)))

train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)

In [ ]:
class Trainer:
    def __init__(
        self,
        model: nn.Module = model,
        learning_rate: float = learning_rate,
        loss_func: nn.Module | None = None,
        train_loader: DataLoader[Tensor] = train_loader,
        val_loader: DataLoader[Tensor] = val_loader,
    ) -> None:
        self.model = model.to(device)
        self.optim = AdamW(model.parameters(), lr=learning_rate)
        self.loss_func = loss_func or MSLELoss()
        self.train_loader = train_loader
        self.val_loader = val_loader

    def train(self, epochs: int, save_every_n_epochs: int | None = None) -> None:
        progress_bar = tqdm(range(epochs))
        digits = math.ceil(math.log10(epochs))
        train_loss = torch.nan
        val_loss = torch.nan
        best_val_loss = torch.inf
        for epoch_idx in progress_bar:
            progress_bar.set_description(f"T: {train_loss:.4f} | V: {val_loss:.4f}")
            train_loss = self.train_loop(epoch_idx).item()
            progress_bar.set_description(f"T: {train_loss:.4f} | V: {val_loss:.4f}")
            val_loss = self.val_loop(epoch_idx).item()
            progress_bar.set_description(f"T: {train_loss:.4f} | V: {val_loss:.4f}")
            if val_loss < best_val_loss:
                self.checkpoint("best_model")
                best_val_loss = val_loss
                mlflow.set_tag("best_epoch", epoch_idx)
                progress_bar.set_description("saving best...")
            if save_every_n_epochs and epoch_idx % save_every_n_epochs == 0:
                self.checkpoint(f"epoch_{epoch_idx:0{digits}}")
                progress_bar.set_description("saving periodic...")

    def train_loop(self, epoch_idx: int) -> Tensor:
        loss = tensor(torch.nan, device=device)
        start_time = time.perf_counter()
        n_samples = 0
        for batch_X, batch_y in self.train_loader:
            self.optim.zero_grad()
            loss = self.run_loss(batch_X.to(device), batch_y.to(device))
            loss.backward()
            self.optim.step()
            n_samples += batch_X.shape[0]
        torch.mps.synchronize()
        sample_rate = n_samples / (time.perf_counter() - start_time)
        mem_allocated = torch.mps.current_allocated_memory() / 1e9
        mlflow.log_metrics(
            {
                "train_loss": loss.item(),
                "sample_rate": sample_rate,
                "mem_allocated": mem_allocated,
            },
            step=epoch_idx,
        )
        return loss

    @torch.inference_mode()
    def val_loop(self, epoch_idx: int) -> Tensor:
        loss = tensor(0., device=device)
        n_batch = 0
        for batch_X, batch_y in self.val_loader:
            loss += self.run_loss(batch_X.to(device), batch_y.to(device))
            n_batch += 1
        loss /= n_batch
        mlflow.log_metrics(
            {
                "val_loss": loss.item(),
            },
            step=epoch_idx,
        )
        return loss

    def run_loss(self, batch_X: Tensor, batch_y: Tensor) -> Tensor:
        model_output = self.model(batch_X)
        return self.loss_func(model_output, batch_y)


    def checkpoint(self, name: str) -> None:
        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp) / "state_dict.pt"
            torch.save(self.model.state_dict(), path)
            mlflow.log_artifact(str(path), artifact_path=f"checkpoints/{name}")


In [ ]:
with mlflow.start_run(
    # run_name="initial_training",
    tags={
        "architecture": "transformer",
        "input_window_days": str(store_data.window_lags),
        "output_window_days": str(store_data.output_lags),
        "device": str(device),
        "git_branch": get_branch(),
        "git_sha": get_sha(),
    },
) as mlflow_run:
    mlflow.log_params(
        {
            "epochs": epochs,
            "learning_rate": learning_rate,
            "split_fraction": split_fraction,
            "batch_size": batch_size,
        }
    )
    train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
    trainer = Trainer(model, train_loader=train_loader, val_loader=val_loader)
    trainer.train(epochs, save_every_n_epochs=save_every_n_epochs)

# Scrap

In [ ]:

# # Batch Size Sweep

# with mlflow.start_run(
#     run_name="batch_size_sweep",
#     tags={
#         "architecture": "transformer",
#         "input_window_days": str(store_data.window_lags),
#         "output_window_days": str(store_data.output_lags),
#         "device": str(device),
#         "git_branch": get_branch(),
#         "git_sha": get_sha(),
#     },
# ) as mlflow_run:
#     mlflow.log_params(
#         {
#             "epochs": epochs,
#             "learning_rate": learning_rate,
#             "split_fraction": split_fraction,
#             # "batch_size": batch_size,
#         }
#     )
#     for batch_size in [64, 128, 256, 512]:
#         with mlflow.start_run(nested=True) as subrun:
#             mlflow.log_param("batch_size", batch_size)
#             train_loader = DataLoader[Tensor](
#                 train_data, batch_size=batch_size, shuffle=True
#             )
#             val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
#             trainer = Trainer(model, train_loader=train_loader, val_loader=val_loader)
#             results_df = trainer.train(epochs)
#             results_df.plot(sharex=True, subplots=True, figsize=(8, 12))